<a href="https://colab.research.google.com/github/priyankamb2725/AI-Projects/blob/main/HeritageGPT_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HeritageGPT – Colab prototype and evaluation
Run the cells top to bottom (**Runtime → Change runtime type → T4 GPU** is recommended).
Each section prints output you can paste into the paper (environment, retrieval metrics, answers, knowledge-graph checks, image results, latency).

> The heritage text in Cell 3 is **starter text written for the prototype**. Replace it with text copied from the UNESCO / IGNCA / Bhuvan pages you cite (keep the URL), then re-run, so the reported results reflect your real sources.

## 1. Install

In [20]:
!pip -q install chromadb sentence-transformers transformers networkx gradio langchain-text-splitters pillow scikit-learn pandas matplotlib

## 2. Imports and environment (paste into Section VI-A of the paper)

In [21]:
import os, time, json, datetime, platform, statistics, zipfile, glob
import numpy as np, pandas as pd, torch, networkx as nx
import chromadb, transformers, sentence_transformers, gradio
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, CLIPModel, CLIPProcessor
from langchain_text_splitters import RecursiveCharacterTextSplitter
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ACCESS_DATE = datetime.date.today().isoformat()

ENV = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "sentence-transformers": sentence_transformers.__version__,
    "chromadb": chromadb.__version__,
    "networkx": nx.__version__,
    "gradio": gradio.__version__,
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "none (CPU)",
}
for k, v in ENV.items():
    print(f"{k:22s}{v}")

python                3.13.16
torch                 2.11.0+cu130
transformers          5.22.0
sentence-transformers 5.7.0
chromadb              1.5.9
networkx              3.7
gradio                6.31.0
device                cuda
gpu                   Tesla T4


## 3. Heritage knowledge collection (replace with your curated source text)

In [22]:
import sys
sys.path.append('/content')

# Dynamically import SITES from heritage_data.py to replace the starter configuration
try:
    from heritage_data import SITES
    print("Successfully loaded curated SITES from heritage_data.py.")
except ImportError as e:
    print(f"Error importing: {e}. Falling back to default configuration.")

print({k: len(v["text"].split()) for k, v in SITES.items()}, "words per site")

Successfully loaded curated SITES from heritage_data.py.
{'Hampi': 99, 'Pattadakal': 98, 'Ajanta Caves': 88, 'Ellora Caves': 89} words per site


In [23]:
with open('/content/heritage_data.py', 'r') as f:
    print(f.read()[:2000])

"""Curated heritage data for HeritageGPT.

The text below is STARTER text. Replace each "text" value with content copied from the
sources you cite (UNESCO, IGNCA, ISRO Bhuvan) and keep the matching title and URL.
"""

SITES = {
    "Hampi": {
        "state": "Karnataka", "lat": 15.3350, "lon": 76.4600,
        "title": "UNESCO World Heritage Centre - Group of Monuments at Hampi",
        "url": "https://whc.unesco.org/en/list/241/",
        "text": (
            "Hampi is a group of monuments in the state of Karnataka, India, on the banks of the "
            "Tungabhadra River. It was the capital of the Vijayanagara Empire, which flourished between "
            "the 14th and 16th centuries. The site includes temples, royal complexes, markets, water "
            "structures and fortifications spread across a boulder-strewn landscape. Notable monuments "
            "include the Virupaksha Temple, which is still in worship, and the Vittala Temple, known for "
            "its stone c

## 4. Knowledge graph (NetworkX)

In [24]:
TRIPLES = [
    ("Hampi", "located_in", "Karnataka"),
    ("Hampi", "associated_with", "Vijayanagara Empire"),
    ("Hampi", "contains", "Virupaksha Temple (Hampi)"),
    ("Hampi", "contains", "Vittala Temple"),
    ("Vittala Temple", "contains", "Stone Chariot"),
    ("Pattadakal", "located_in", "Karnataka"),
    ("Pattadakal", "associated_with", "Chalukya dynasty"),
    ("Pattadakal", "contains", "Virupaksha Temple (Pattadakal)"),
    ("Pattadakal", "contains", "Sangameshwara Temple"),
    ("Pattadakal", "contains", "Papanatha Temple"),
    ("Pattadakal", "related_to", "Nagara architecture"),
    ("Pattadakal", "related_to", "Dravidian architecture"),
    ("Ajanta Caves", "located_in", "Maharashtra"),
    ("Ajanta Caves", "related_to", "Buddhism"),
    ("Ajanta Caves", "related_to", "Rock-cut architecture"),
    ("Ajanta Caves", "contains", "Viharas"),
    ("Ajanta Caves", "contains", "Chaityas"),
    ("Ellora Caves", "located_in", "Maharashtra"),
    ("Ellora Caves", "related_to", "Buddhism"),
    ("Ellora Caves", "related_to", "Hinduism"),
    ("Ellora Caves", "related_to", "Jainism"),
    ("Ellora Caves", "related_to", "Rock-cut architecture"),
    ("Ellora Caves", "contains", "Kailasa Temple"),
    ("Kailasa Temple", "associated_with", "Rashtrakuta dynasty"),
]
KG = nx.DiGraph()
for s, r, o in TRIPLES:
    KG.add_edge(s, o, relation=r)

def _objs(node, rel):  return {o for _, o, d in KG.out_edges(node, data=True) if d["relation"] == rel}
def _subs(node, rel):  return {s for s, _, d in KG.in_edges(node, data=True) if d["relation"] == rel}

def state_of(site):        return next(iter(_objs(site, "located_in")), None)
def monuments_of(site):    return _objs(site, "contains")
def entities_of(node):     return {o for _, o, _ in KG.out_edges(node, data=True)}
def sites_in(region):      return _subs(region, "located_in")

def kg_facts(site):
    lines = [f"{s} {d['relation'].replace('_', ' ')} {o}." for s, o, d in KG.out_edges(site, data=True)]
    for m in monuments_of(site):
        lines += [f"{s} {d['relation'].replace('_', ' ')} {o}." for s, o, d in KG.out_edges(m, data=True)]
    return lines

print("KG nodes:", KG.number_of_nodes(), "| edges:", KG.number_of_edges())
print(kg_facts("Ellora Caves"))

KG nodes: 24 | edges: 24
['Ellora Caves located in Maharashtra.', 'Ellora Caves related to Buddhism.', 'Ellora Caves related to Hinduism.', 'Ellora Caves related to Jainism.', 'Ellora Caves related to Rock-cut architecture.', 'Ellora Caves contains Kailasa Temple.', 'Kailasa Temple associated with Rashtrakuta dynasty.']


## 5. Index documents (Sentence Transformers + ChromaDB)

In [25]:
splitter = RecursiveCharacterTextSplitter(chunk_size=350, chunk_overlap=60)
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)

client = chromadb.Client()
try: client.delete_collection("heritage")
except Exception: pass
col = client.create_collection("heritage", metadata={"hnsw:space": "cosine"})

ids, docs, metas = [], [], []
for site, d in SITES.items():
    for i, ch in enumerate(splitter.split_text(d["text"])):
        ids.append(f"{site}-{i}"); docs.append(ch)
        metas.append({"site": site, "title": d["title"], "url": d["url"], "access_date": ACCESS_DATE, "chunk": i})
col.add(ids=ids, documents=docs, metadatas=metas,
        embeddings=embedder.encode(docs, normalize_embeddings=True).tolist())
print(f"Indexed {len(docs)} chunks from {len(SITES)} sites using curated SITES data")

for h in retrieve("Which empire ruled from Hampi?", k=3):
    print(round(1 - h["distance"], 3), h["meta"]["site"], "|", h["text"][:90], "...")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Indexed 8 chunks from 4 sites using curated SITES data
0.56 Hampi | Hampi is a group of monuments in the state of Karnataka, India, on the banks of the Tungab ...
0.41 Hampi | spread across a boulder-strewn landscape. Notable monuments include the Virupaksha Temple, ...
0.19 Ajanta Caves | about 30 caves, including viharas, which were monasteries, and chaityas, which were prayer ...


## 6. Language model (FLAN-T5) and CLIP

In [26]:
LM_NAME = "google/flan-t5-base"        # use "google/flan-t5-large" on a GPU for better answers
tok = AutoTokenizer.from_pretrained(LM_NAME)
lm = AutoModelForSeq2SeqLM.from_pretrained(LM_NAME).to(DEVICE).eval()
ABSTAIN = "I do not have enough information."

def generate(question, context, max_new_tokens=200):
    prompt = ("Answer the question using only the context below. "
              f"If the context does not contain the answer, reply exactly: {ABSTAIN}\n\n"
              f"Context:\n{context}\n\nQuestion: {question}\nAnswer:")
    inp = tok(prompt, return_tensors="pt", truncation=True, max_length=1024).to(DEVICE)
    with torch.no_grad():
        out = lm.generate(**inp, max_new_tokens=max_new_tokens, num_beams=4, early_stopping=True)
    return tok.decode(out[0], skip_special_tokens=True)

CLIP_NAME = "openai/clip-vit-base-patch32"
clip = CLIPModel.from_pretrained(CLIP_NAME).to(DEVICE).eval()
clip_proc = CLIPProcessor.from_pretrained(CLIP_NAME)

CLIP_PROMPTS = {
    "Hampi": ["a photo of the ruins of Hampi in Karnataka, India",
              "a photo of the stone chariot at the Vittala Temple, Hampi",
              "a photo of Vijayanagara Empire temples among boulders"],
    "Pattadakal": ["a photo of the Pattadakal temples in Karnataka, India",
                   "a photo of the Virupaksha Temple at Pattadakal",
                   "a photo of Chalukya dynasty temples"],
    "Ajanta Caves": ["a photo of the Ajanta Caves in Maharashtra, India",
                     "a photo of a rock-cut Buddhist cave with paintings",
                     "a photo of a Buddhist monastery cave at Ajanta"],
    "Ellora Caves": ["a photo of the Ellora Caves in Maharashtra, India",
                     "a photo of the Kailasa Temple carved from a single rock",
                     "a photo of a rock-cut Hindu, Buddhist or Jain cave at Ellora"],
}
SITE_NAMES = list(SITES.keys())

def _t(x):  # works across transformers versions
    return x if torch.is_tensor(x) else getattr(x, "pooler_output", x[0])

def _text_feats():
    feats = []
    for s in SITE_NAMES:
        t = clip_proc(text=CLIP_PROMPTS[s], return_tensors="pt", padding=True).to(DEVICE)
        with torch.no_grad(): f = _t(clip.get_text_features(**t))
        f = f / f.norm(dim=-1, keepdim=True)
        f = f.mean(0); feats.append(f / f.norm())
    return torch.stack(feats)
TEXT_FEATS = _text_feats()

def identify(img):
    inp = clip_proc(images=img.convert("RGB"), return_tensors="pt").to(DEVICE)
    with torch.no_grad(): f = _t(clip.get_image_features(**inp))
    f = f / f.norm(dim=-1, keepdim=True)
    p = (100 * f @ TEXT_FEATS.T).softmax(-1)[0].cpu().numpy()
    return sorted(zip(SITE_NAMES, p.tolist()), key=lambda x: -x[1])
print("Models loaded on", DEVICE)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Models loaded on cuda


## 7. End-to-end pipeline

In [27]:
def ask(question=None, image=None, k=4):
    t0 = time.perf_counter()
    ranks = identify(image) if image is not None else None
    pred = ranks[0][0] if ranks else None
    question = question or "Tell me about this monument."
    hits = retrieve(question, k=k, site=pred)
    t1 = time.perf_counter()

    site = pred or hits[0]["meta"]["site"]
    passages = "\n".join(f"[{i+1}] {h['text']}" for i, h in enumerate(hits))
    context = passages + "\nKnowledge graph facts: " + " ".join(kg_facts(site))
    answer = generate(question, context)
    t2 = time.perf_counter()

    seen, sources = set(), []
    for h in hits:
        if h["meta"]["url"] not in seen:
            seen.add(h["meta"]["url"]); sources.append((h["meta"]["title"], h["meta"]["url"]))
    s = SITES[site]
    return {"question": question, "answer": answer, "site": site, "image_ranks": ranks,
            "hits": hits, "sources": sources, "coords": (s["lat"], s["lon"], s["state"]),
            "retrieval_s": t1 - t0, "generation_s": t2 - t1, "total_s": t2 - t0}

def show(r):
    print("Q:", r["question"]); print("A:", r["answer"])
    print("Site:", r["site"], "| State:", r["coords"][2], "| Lat/Lon:", r["coords"][:2])
    if r["image_ranks"]: print("CLIP ranking:", [(n, round(p, 3)) for n, p in r["image_ranks"]])
    for t, u in r["sources"]: print("Source:", t, "-", u)
    print(f"Latency: retrieval {r['retrieval_s']:.2f}s, generation {r['generation_s']:.2f}s\n")

for q in ["Tell me about Hampi", "Which dynasty built the temples at Pattadakal?",
          "What religions are represented at Ellora?", "What is the ticket price for the Ajanta Caves?"]:
    show(ask(q))

Q: Tell me about Hampi
A: Hampi is a group of monuments in the state of Karnataka, India
Site: Hampi | State: Karnataka | Lat/Lon: (15.335, 76.46)
Source: UNESCO World Heritage Centre - Group of Monuments at Hampi - https://whc.unesco.org/en/list/241/
Source: UNESCO World Heritage Centre - Ajanta Caves - https://whc.unesco.org/en/list/242/
Latency: retrieval 0.02s, generation 0.60s

Q: Which dynasty built the temples at Pattadakal?
A: Chalukya
Site: Pattadakal | State: Karnataka | Lat/Lon: (15.9486, 75.8159)
Source: UNESCO World Heritage Centre - Group of Monuments at Pattadakal - https://whc.unesco.org/en/list/239/
Source: UNESCO World Heritage Centre - Group of Monuments at Hampi - https://whc.unesco.org/en/list/241/
Source: UNESCO World Heritage Centre - Ellora Caves - https://whc.unesco.org/en/list/243/
Latency: retrieval 0.01s, generation 0.26s

Q: What religions are represented at Ellora?
A: Buddhist, Hindu and Jain
Site: Ellora Caves | State: Maharashtra | Lat/Lon: (20.0258, 75.

## 8. Retrieval evaluation – Precision@k, Recall@k, MRR (Section VIII-A)
Gold labels: a chunk is *relevant* if it belongs to the expected site **and** contains an expected keyword. Edit `TEST_SET` to match your own sources.

In [28]:
TEST_SET = [
    ("Which river is Hampi located on?", "Hampi", ["tungabhadra"]),
    ("Which empire had its capital at Hampi?", "Hampi", ["vijayanagara"]),
    ("What is the Vittala Temple known for?", "Hampi", ["stone chariot"]),
    ("Which dynasty is associated with Pattadakal?", "Pattadakal", ["chalukya"]),
    ("Who built the Virupaksha Temple at Pattadakal?", "Pattadakal", ["lokamahadevi"]),
    ("Which architectural styles are combined at Pattadakal?", "Pattadakal", ["nagara", "dravidian"]),
    ("What type of monuments are the Ajanta Caves?", "Ajanta Caves", ["rock-cut"]),
    ("Which river is near the Ajanta Caves?", "Ajanta Caves", ["waghora"]),
    ("In which centuries were the Ajanta Caves created?", "Ajanta Caves", ["2nd century", "5th century"]),
    ("How many monasteries and temples are at Ellora?", "Ellora Caves", ["34"]),
    ("Which religions are represented at Ellora?", "Ellora Caves", ["buddhist", "hindu", "jain"]),
    ("Which temple at Ellora is carved from a single rock?", "Ellora Caves", ["kailasa"]),
]
UNANSWERABLE = ["What is the ticket price for the Ajanta Caves?",
                "Who won the 2011 cricket world cup?",
                "What are the opening hours of the Hampi bazaar?"]

def is_rel(text, meta, site, kws):
    return meta["site"] == site and any(w in text.lower() for w in kws)

KS = (1, 3, 5)
rows = []
for q, site, kws in TEST_SET:
    total_rel = sum(is_rel(d, m, site, kws) for d, m in zip(docs, metas))
    hits = retrieve(q, k=max(KS))
    flags = [is_rel(h["text"], h["meta"], site, kws) for h in hits]
    row = {"question": q, "gold_site": site, "total_relevant": total_rel}
    for k in KS:
        row[f"P@{k}"] = sum(flags[:k]) / k
        row[f"R@{k}"] = sum(flags[:k]) / total_rel if total_rel else float("nan")
    row["RR"] = next((1 / (i + 1) for i, f in enumerate(flags) if f), 0.0)
    rows.append(row)

ret_df = pd.DataFrame(rows)
summary = {**{f"Precision@{k}": ret_df[f"P@{k}"].mean() for k in KS},
           **{f"Recall@{k}": ret_df[f"R@{k}"].mean() for k in KS}, "MRR": ret_df["RR"].mean()}
ret_summary = pd.Series(summary).round(3)
print(f"Retrieval evaluation on {len(TEST_SET)} questions, {len(docs)} chunks")
print(ret_summary.to_string())
ret_df.round(3)

Retrieval evaluation on 12 questions, 8 chunks
Precision@1    1.000
Precision@3    0.389
Precision@5    0.233
Recall@1       0.917
Recall@3       1.000
Recall@5       1.000
MRR            1.000


,question,gold_site,total_relevant,P@1,R@1,P@3,R@3,P@5,R@5,RR
0,Which river is Hampi located on?,Hampi,1,1.0,1.0,0.333,1.0,0.2,1.0,1.0
1,Which empire had its capital at Hampi?,Hampi,1,1.0,1.0,0.333,1.0,0.2,1.0,1.0
2,What is the Vittala Temple known for?,Hampi,1,1.0,1.0,0.333,1.0,0.2,1.0,1.0
3,Which dynasty is associated with Pattadakal?,Pattadakal,1,1.0,1.0,0.333,1.0,0.2,1.0,1.0
4,Who built the Virupaksha Temple at Pattadakal?,Pattadakal,1,1.0,1.0,0.333,1.0,0.2,1.0,1.0
5,Which architectural styles are combined at Pat...,Pattadakal,2,1.0,0.5,0.667,1.0,0.4,1.0,1.0
6,What type of monuments are the Ajanta Caves?,Ajanta Caves,1,1.0,1.0,0.333,1.0,0.2,1.0,1.0
7,Which river is near the Ajanta Caves?,Ajanta Caves,1,1.0,1.0,0.333,1.0,0.2,1.0,1.0
8,In which centuries were the Ajanta Caves created?,Ajanta Caves,1,1.0,1.0,0.333,1.0,0.2,1.0,1.0
9,How many monasteries and temples are at Ellora?,Ellora Caves,1,1.0,1.0,0.333,1.0,0.2,1.0,1.0


## 9. Response evaluation (Section VIII-B)
Automatic proxies (keyword match, source match, abstention) plus a CSV with blank columns for **human** scoring of factuality, relevance, completeness and unsupported statements.

In [29]:
resp_rows, lat = [], []
for q, site, kws in TEST_SET:
    r = ask(q); lat.append(r)
    resp_rows.append({
        "question": q, "answer": r["answer"], "gold_site": site,
        "top_source_site_correct": r["hits"][0]["meta"]["site"] == site,
        "keyword_match": any(w in r["answer"].lower() for w in kws),
        "has_source": len(r["sources"]) > 0,
        "human_factual(0/1)": "", "human_relevant(0/1)": "", "human_complete(0/1)": "", "human_unsupported(0/1)": "",
    })
resp_df = pd.DataFrame(resp_rows)

abst = []
for q in UNANSWERABLE:
    r = ask(q); lat.append(r)
    abst.append({"question": q, "answer": r["answer"], "abstained": ABSTAIN.lower().rstrip(".") in r["answer"].lower()})
abst_df = pd.DataFrame(abst)

resp_summary = {
    "answerable_questions": len(resp_df),
    "keyword_match_rate": round(resp_df["keyword_match"].mean(), 3),
    "top_source_correct_rate": round(resp_df["top_source_site_correct"].mean(), 3),
    "answers_with_source_link": round(resp_df["has_source"].mean(), 3),
    "unanswerable_questions": len(abst_df),
    "abstention_rate": round(abst_df["abstained"].mean(), 3),
}
print(json.dumps(resp_summary, indent=2))
resp_df

{
  "answerable_questions": 12,
  "keyword_match_rate": 0.75,
  "top_source_correct_rate": 1.0,
  "answers_with_source_link": 1.0,
  "unanswerable_questions": 3,
  "abstention_rate": 0.0
}


,question,answer,gold_site,top_source_site_correct,keyword_match,has_source,human_factual(0/1),human_relevant(0/1),human_complete(0/1),human_unsupported(0/1)
0,Which river is Hampi located on?,Tungabhadra River,Hampi,True,True,True,,,,
1,Which empire had its capital at Hampi?,Vijayanagara Empire,Hampi,True,True,True,,,,
2,What is the Vittala Temple known for?,stone chariot and musical pillars,Hampi,True,True,True,,,,
3,Which dynasty is associated with Pattadakal?,Chalukya,Pattadakal,True,True,True,,,,
4,Who built the Virupaksha Temple at Pattadakal?,Queen Lokamahadevi,Pattadakal,True,True,True,,,,
5,Which architectural styles are combined at Pat...,northern and southern India,Pattadakal,True,False,True,,,,
6,What type of monuments are the Ajanta Caves?,Buddhist cave monuments,Ajanta Caves,True,False,True,,,,
7,Which river is near the Ajanta Caves?,Waghora River,Ajanta Caves,True,True,True,,,,
8,In which centuries were the Ajanta Caves created?,[1],Ajanta Caves,True,False,True,,,,
9,How many monasteries and temples are at Ellora?,34,Ellora Caves,True,True,True,,,,


In [30]:
abst_df

,question,answer,abstained
0,What is the ticket price for the Ajanta Caves?,not enough information,False
1,Who won the 2011 cricket world cup?,India,False
2,What are the opening hours of the Hampi bazaar?,not enough information,False


## 10. Knowledge-graph evaluation – competency questions (Section VIII-D)

In [31]:
KG_TESTS = [
    ("Which state is Hampi located in?",            lambda: state_of("Hampi"),               "Karnataka"),
    ("Which state is Ajanta Caves located in?",     lambda: state_of("Ajanta Caves"),        "Maharashtra"),
    ("Which sites are located in Maharashtra?",     lambda: sites_in("Maharashtra"),         {"Ajanta Caves", "Ellora Caves"}),
    ("Which sites are located in Karnataka?",       lambda: sites_in("Karnataka"),           {"Hampi", "Pattadakal"}),
    ("Which monuments are at Pattadakal?",          lambda: monuments_of("Pattadakal"),      {"Virupaksha Temple (Pattadakal)", "Sangameshwara Temple", "Papanatha Temple"}),
    ("Which entities connect to Kailasa Temple?",   lambda: entities_of("Kailasa Temple"),   {"Rashtrakuta dynasty"}),
]
kg_rows = []
for q, fn, exp in KG_TESTS:
    got = fn()
    kg_rows.append({"competency_question": q, "expected": str(exp), "returned": str(got), "correct": got == exp})
kg_df = pd.DataFrame(kg_rows)
kg_summary = {"nodes": KG.number_of_nodes(), "edges": KG.number_of_edges(),
              "competency_pass_rate": round(kg_df["correct"].mean(), 3)}
print(kg_summary)
kg_df

{'nodes': 24, 'edges': 24, 'competency_pass_rate': np.float64(1.0)}


,competency_question,expected,returned,correct
0,Which state is Hampi located in?,Karnataka,Karnataka,True
1,Which state is Ajanta Caves located in?,Maharashtra,Maharashtra,True
2,Which sites are located in Maharashtra?,"{'Ellora Caves', 'Ajanta Caves'}","{'Ellora Caves', 'Ajanta Caves'}",True
3,Which sites are located in Karnataka?,"{'Hampi', 'Pattadakal'}","{'Hampi', 'Pattadakal'}",True
4,Which monuments are at Pattadakal?,"{'Papanatha Temple', 'Sangameshwara Temple', '...","{'Papanatha Temple', 'Sangameshwara Temple', '...",True
5,Which entities connect to Kailasa Temple?,{'Rashtrakuta dynasty'},{'Rashtrakuta dynasty'},True


## 11. Image-recognition evaluation (Section VIII-C)
Create a folder `/content/images/<site name>/` for each class, using exactly these folder names: `Hampi`, `Pattadakal`, `Ajanta Caves`, `Ellora Caves`.
Either upload a **zip** with that structure in the next cell, or place images there by hand. Use only properly licensed images (e.g. your own photos, Wikimedia Commons, or IHIRD if you have access) and report the source and counts in the paper.

In [32]:
from google.colab import files
up = files.upload()   # choose a .zip with the class folders (skip this cell if you added images manually)
for name in up:
    if name.endswith(".zip"):
        zipfile.ZipFile(name).extractall("/content/images")
print(os.listdir("/content/images") if os.path.exists("/content/images") else "no images folder yet")

Saving Screenshot 2026-11-03 232315.png to Screenshot 2026-11-03 232315.png
no images folder yet


In [33]:
from sklearn.metrics import confusion_matrix, classification_report
IMG_ROOT = "/content/images"
y_true, y_pred, top3_ok = [], [], []
for site in SITE_NAMES:
    for p in sorted(glob.glob(os.path.join(IMG_ROOT, "**", site, "*.*"), recursive=True) + glob.glob(os.path.join(IMG_ROOT, site, "*.*"))):
        if not p.lower().endswith((".jpg", ".jpeg", ".png", ".webp")): continue
        ranks = identify(Image.open(p))
        y_true.append(site); y_pred.append(ranks[0][0]); top3_ok.append(site in [n for n, _ in ranks[:3]])

if y_true:
    top1 = np.mean(np.array(y_true) == np.array(y_pred))
    img_summary = {"images": len(y_true), "top1_accuracy": round(float(top1), 3), "top3_accuracy": round(float(np.mean(top3_ok)), 3)}
    print(img_summary)
    cm = pd.DataFrame(confusion_matrix(y_true, y_pred, labels=SITE_NAMES),
                      index=[f"true: {s}" for s in SITE_NAMES], columns=[f"pred: {s}" for s in SITE_NAMES])
    print(cm.to_string())
    print(classification_report(y_true, y_pred, labels=SITE_NAMES, zero_division=0))
else:
    img_summary = None
    print("No images found - add images to /content/images/<site name>/ and re-run. Do not report image accuracy until this has run.")

No images found - add images to /content/images/<site name>/ and re-run. Do not report image accuracy until this has run.


## 12. Latency (Section IX)

In [34]:
lat_df = pd.DataFrame([{k: r[k] for k in ("retrieval_s", "generation_s", "total_s")} for r in lat])
lat_summary = lat_df.agg(["mean", "median", "max"]).round(3)
print(f"Over {len(lat_df)} text queries on {ENV['gpu']}:")
print(lat_summary.to_string())

Over 15 text queries on Tesla T4:
        retrieval_s  generation_s  total_s
mean          0.010         0.210    0.220
median        0.010         0.206    0.217
max           0.012         0.310    0.319


## 13. Interactive demo (take screenshots for the paper)

In [35]:
import gradio as gr

def ui(question, image):
    if not question and image is None:
        return "Please type a question or upload a monument image.", "", ""
    r = ask(question or None, image)
    lat_, lon_, state = r["coords"]
    src = "\n".join(f"- [{t}]({u})" for t, u in r["sources"])
    maps = f"https://www.google.com/maps?q={lat_},{lon_}"
    info = f"**Site:** {r['site']} ({state})  \n**Map:** [{lat_}, {lon_}]({maps})  \n\n**Sources**\n{src}"
    pred = "\n".join(f"{n}: {p:.2f}" for n, p in r["image_ranks"]) if r["image_ranks"] else "No image supplied"
    return r["answer"], info, pred

demo = gr.Interface(
    fn=ui,
    inputs=[gr.Textbox(label="Question", placeholder="Tell me about Hampi"), gr.Image(type="pil", label="Monument image (optional)")],
    outputs=[gr.Textbox(label="Answer"), gr.Markdown(label="Site, map and sources"), gr.Textbox(label="CLIP candidate identification")],
    title="HeritageGPT", description="Prototype heritage assistant. Image predictions are candidate identifications only.")
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2019f4a6afe2952899.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 14. Export everything for the paper

In [36]:
ret_df.to_csv("retrieval_per_question.csv", index=False)
resp_df.to_csv("response_eval_for_human_scoring.csv", index=False)
abst_df.to_csv("abstention_eval.csv", index=False)
kg_df.to_csv("kg_competency_eval.csv", index=False)

report = {"environment": ENV, "models": {"embedding": "all-MiniLM-L6-v2", "generator": LM_NAME, "vision": CLIP_NAME},
          "corpus": {"sites": len(SITES), "chunks": len(docs)},
          "retrieval": ret_summary.to_dict(), "response": resp_summary, "knowledge_graph": kg_summary,
          "image": img_summary, "latency_seconds": lat_summary.to_dict()}
with open("heritagegpt_results.json", "w") as f: json.dump(report, f, indent=2, default=str)
print(json.dumps(report, indent=2, default=str))

from google.colab import files
for fn in ["heritagegpt_results.json", "retrieval_per_question.csv", "response_eval_for_human_scoring.csv",
           "abstention_eval.csv", "kg_competency_eval.csv"]:
    files.download(fn)

{
  "environment": {
    "python": "3.13.16",
    "torch": "2.11.0+cu130",
    "transformers": "5.22.0",
    "sentence-transformers": "5.7.0",
    "chromadb": "1.5.9",
    "networkx": "3.7",
    "gradio": "6.31.0",
    "device": "cuda",
    "gpu": "Tesla T4"
  },
  "models": {
    "embedding": "all-MiniLM-L6-v2",
    "generator": "google/flan-t5-base",
    "vision": "openai/clip-vit-base-patch32"
  },
  "corpus": {
    "sites": 4,
    "chunks": 8
  },
  "retrieval": {
    "Precision@1": 1.0,
    "Precision@3": 0.389,
    "Precision@5": 0.233,
    "Recall@1": 0.917,
    "Recall@3": 1.0,
    "Recall@5": 1.0,
    "MRR": 1.0
  },
  "response": {
    "answerable_questions": 12,
    "keyword_match_rate": 0.75,
    "top_source_correct_rate": 1.0,
    "answers_with_source_link": 1.0,
    "unanswerable_questions": 3,
    "abstention_rate": 0.0
  },
  "knowledge_graph": {
    "nodes": 24,
    "edges": 24,
    "competency_pass_rate": 1.0
  },
  "image": null,
  "latency_seconds": {
    "retrieval

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>